# Identity Clues Shift

This notebook studies how identity framing changes over time in ads from the main parties, using two source definitions:
- `page_name`
- `funding_entity`

The focus is on in-group language (`we`, `our`) versus out-group language (`they`, `them`, opponent references), and on how those cues shift across weeks and parties.


In [ ]:
%load_ext watermark
%watermark -v -n -m -p numpy,scipy,sklearn,pandas

import warnings
import os
import sys
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

%load_ext autoreload
%autoreload 2
%matplotlib inline

print(f'default sys.path: {sys.path}')

PROJ_ROOT = os.path.abspath(os.path.join(os.pardir))
PROJ_ROOT = os.path.abspath(os.path.join(PROJ_ROOT, os.pardir))
sys.path.append(PROJ_ROOT)
print(f'Project root: {PROJ_ROOT}')

from utils import mpl_settings
from utils.dataset_utilities import load_data

mpl_settings.apply_settings()
plt.style.use(mpl_settings.plot_settings)
sns.set_theme(style='whitegrid')


## Data Loading And Main-Party Filter

The notebook always filters to the main party set used in `party_analysis.ipynb`.


In [ ]:
file_path = '../../data/processed/2022_aus_elections_mar_to_may.csv'
df = load_data(file_path)

main_parties = ['Liberal', 'Labor', 'Green', 'Independent', 'United Australia Party']
df = df[df['macro_party_uap'].isin(main_parties)].copy()

df['ad_creation_time'] = pd.to_datetime(df['ad_creation_time'], errors='coerce')
df['ad_delivery_start_time'] = pd.to_datetime(df['ad_delivery_start_time'], errors='coerce')
df['ad_delivery_stop_time'] = pd.to_datetime(df['ad_delivery_stop_time'], errors='coerce')
df['ad_date'] = df['ad_delivery_start_time'].fillna(df['ad_creation_time'])

stop_time = df['ad_delivery_stop_time'].fillna(df['ad_delivery_start_time'])
df['duration_days'] = (stop_time - df['ad_delivery_start_time']).dt.days.add(1)
df['duration_days'] = df['duration_days'].clip(lower=1)
df.loc[df['ad_delivery_start_time'].isna(), 'duration_days'] = np.nan

df['impressions_per_day'] = df['mean_impressions'] / df['duration_days']
df['week_start'] = df['ad_date'].dt.to_period('W').dt.start_time

df['ad_text'] = (
    df['ad_creative_body']
    .fillna(df['multiword_safe_lemmatized'])
    .fillna('')
    .astype(str)
)
df['ad_text_clean'] = (
    df['ad_text']
    .str.lower()
    .str.replace(r'http\S+|www\S+|https\S+', ' ', regex=True)
    .str.replace(r'[^a-z0-9\s]', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

print('Filtered rows:', len(df))
print('Ads with a source page_name:', df['page_name'].notna().sum())
print('Ads with a source funding_entity:', df['funding_entity'].notna().sum())
print('Date coverage:', df['ad_date'].min(), 'to', df['ad_date'].max())


## Identity Lexicons

The lexicons are intentionally small and interpretable. They mix single-word cues with common multi-word phrases that signal group alignment or opposition.


In [ ]:
IN_GROUP_TERMS = [
    r'\bwe\b',
    r'\bus\b',
    r'\bour\b',
    r'\bours\b',
    r'\bourselves\b',
    r'\bwe are\b',
    r'\bwe will\b',
    r'\bwe can\b',
    r'\bwe need\b',
    r'\bour team\b',
    r'\bwith us\b',
    r'\btogether\b',
    r'\bone team\b',
    r'\bour community\b',
    r'\bour future\b',
]

OUT_GROUP_TERMS = [
    r'\bthey\b',
    r'\bthem\b',
    r'\btheir\b',
    r'\btheirs\b',
    r'\bthemselves\b',
    r'\bother side\b',
    r'\bopponent\w*\b',
    r'\bopposition\b',
    r'\bthe government\b',
    r'\bthe liberals?\b',
    r'\bthe labor party\b',
    r'\bthe labour party\b',
    r'\bthe greens?\b',
    r'\bunited australia party\b',
    r'\bclive palmer\b',
    r'\banthony albanese\b',
    r'\bscott morrison\b',
    r'\badam bandt\b',
]

def compile_terms(terms):
    return [(term, re.compile(term, flags=re.IGNORECASE)) for term in terms]

IN_GROUP_PATTERNS = compile_terms(IN_GROUP_TERMS)
OUT_GROUP_PATTERNS = compile_terms(OUT_GROUP_TERMS)

def count_hits(text, compiled_terms):
    if not isinstance(text, str) or not text:
        return 0
    return sum(len(pattern.findall(text)) for _, pattern in compiled_terms)

def classify_identity(text):
    ingroup = count_hits(text, IN_GROUP_PATTERNS)
    outgroup = count_hits(text, OUT_GROUP_PATTERNS)
    total = ingroup + outgroup
    balance = (ingroup - outgroup) / total if total else np.nan
    polarity = abs(balance) if pd.notna(balance) else np.nan
    return pd.Series({
        'ingroup_count': ingroup,
        'outgroup_count': outgroup,
        'identity_total': total,
        'ingroup_ratio': ingroup / total if total else np.nan,
        'outgroup_ratio': outgroup / total if total else np.nan,
        'balance_index': balance,
        'polarization_index': polarity,
        'has_identity_cue': total > 0,
    })

def summarize_source_shift(group):
    group = group.sort_values('week_start').copy()
    valid = group['balance_index'].dropna()
    if len(valid) == 0:
        slope = np.nan
        balance_range = np.nan
        mean_balance = np.nan
        mean_polarization = np.nan
    else:
        y = valid.to_numpy(dtype=float)
        x = np.arange(len(y), dtype=float)
        slope = np.polyfit(x, y, 1)[0] if len(y) >= 2 else np.nan
        balance_range = float(np.nanmax(y) - np.nanmin(y)) if len(y) >= 1 else np.nan
        mean_balance = float(np.nanmean(y))
        mean_polarization = float(group['polarization_index'].mean())
    return pd.Series({
        'weeks_active': int(group['week_start'].nunique()),
        'ads': int(group['id'].count()),
        'mean_balance': mean_balance,
        'balance_min': float(np.nanmin(valid)) if len(valid) else np.nan,
        'balance_max': float(np.nanmax(valid)) if len(valid) else np.nan,
        'balance_range': balance_range,
        'slope_per_week': slope,
        'mean_polarization': mean_polarization,
        'mean_identity_total': float(group['identity_total'].mean()),
        'identity_ad_share': float(group['has_identity_cue'].mean()),
    })

def analyze_source_definition(frame, source_col):
    data = frame.loc[frame[source_col].notna()].copy()
    data = data[data[source_col].astype(str).str.strip().ne('')].copy()
    data[source_col] = data[source_col].astype(str).str.strip()

    metrics = data['ad_text_clean'].apply(classify_identity)
    data = pd.concat([data.reset_index(drop=True), metrics.reset_index(drop=True)], axis=1)

    weekly_party = (
        data.groupby(['week_start', 'macro_party_uap'], dropna=False)
        .agg(
            ads=('id', 'size'),
            mean_ingroup_ratio=('ingroup_ratio', 'mean'),
            mean_outgroup_ratio=('outgroup_ratio', 'mean'),
            mean_balance=('balance_index', 'mean'),
            mean_polarization=('polarization_index', 'mean'),
            identity_ad_share=('has_identity_cue', 'mean'),
        )
        .reset_index()
        .sort_values(['macro_party_uap', 'week_start'])
    )

    source_weekly = (
        data.groupby([source_col, 'week_start'], dropna=False)
        .agg(
            ads=('id', 'size'),
            mean_ingroup_ratio=('ingroup_ratio', 'mean'),
            mean_outgroup_ratio=('outgroup_ratio', 'mean'),
            mean_balance=('balance_index', 'mean'),
            mean_polarization=('polarization_index', 'mean'),
            identity_ad_share=('has_identity_cue', 'mean'),
        )
        .reset_index()
        .sort_values([source_col, 'week_start'])
    )

    source_summary = (
        source_weekly.groupby(source_col, dropna=False)
        .apply(summarize_source_shift)
        .reset_index()
        .sort_values(['balance_range', 'slope_per_week', 'ads'], ascending=[False, False, False])
    )

    party_summary = (
        data.groupby('macro_party_uap', dropna=False)
        .agg(
            ads=('id', 'size'),
            identity_ad_share=('has_identity_cue', 'mean'),
            mean_ingroup_count=('ingroup_count', 'mean'),
            mean_outgroup_count=('outgroup_count', 'mean'),
            mean_balance=('balance_index', 'mean'),
            mean_polarization=('polarization_index', 'mean'),
            mean_identity_total=('identity_total', 'mean'),
        )
        .reset_index()
        .sort_values('mean_balance', ascending=False)
    )

    return {
        'source_col': source_col,
        'data': data,
        'weekly_party': weekly_party,
        'source_weekly': source_weekly,
        'source_summary': source_summary,
        'party_summary': party_summary,
    }

print('In-group terms:', len(IN_GROUP_TERMS))
print('Out-group terms:', len(OUT_GROUP_TERMS))


## Run The Analysis Twice

The same logic is applied to source identity defined as `page_name` and as `funding_entity`.


In [ ]:
analyses = {
    'page_name': analyze_source_definition(df, 'page_name'),
    'funding_entity': analyze_source_definition(df, 'funding_entity'),
}

combined_party_summary = pd.concat(
    {
        key: value['party_summary'].set_index('macro_party_uap')
        for key, value in analyses.items()
    },
    names=['source_definition', 'macro_party_uap'],
).reset_index()

for source_name, result in analyses.items():
    print()
    print('=' * 80)
    print(f'Source definition: {source_name}')
    print('Ads analysed:', len(result['data']))
    print()
    print('Party-level summary:')
    print(result['party_summary'].round(4).to_string(index=False))
    print()
    print('Top source shifts:')
    cols = [source_name, 'ads', 'weeks_active', 'mean_balance', 'balance_range', 'slope_per_week', 'identity_ad_share']
    print(result['source_summary'][cols].head(10).round(4).to_string(index=False))

print()
print('Combined party summary preview:')
print(combined_party_summary.round(4).head(20).to_string(index=False))


## Weekly Identity Balance Over Time

Positive values mean more in-group framing; negative values mean more out-group framing.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=True)

for ax, (source_name, result) in zip(axes, analyses.items()):
    weekly = result['weekly_party'].dropna(subset=['week_start']).copy()
    sns.lineplot(
        data=weekly,
        x='week_start',
        y='mean_balance',
        hue='macro_party_uap',
        marker='o',
        ax=ax,
    )
    ax.axhline(0, color='black', linewidth=1, alpha=0.7)
    ax.set_title(f'Weekly mean balance by party - {source_name}')
    ax.set_xlabel('Week')
    ax.set_ylabel('Mean balance index')
    ax.tick_params(axis='x', rotation=45)
    ax.legend(title='Party', fontsize=9, title_fontsize=10, loc='best')

plt.tight_layout()
plt.show()


## Largest Source Shifts

These plots rank sources by the range of their weekly identity balance. This surfaces pages or funding entities whose framing changed most over the campaign.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, (source_name, result) in zip(axes, analyses.items()):
    plot_df = result['source_summary'].head(10).copy()
    plot_df['label'] = plot_df[source_name].astype(str).str.replace(r'\s+', ' ', regex=True).str.slice(0, 60)
    colors = np.where(plot_df['slope_per_week'] >= 0, '#1f77b4', '#d62728')
    sns.barplot(data=plot_df, y='label', x='balance_range', palette=colors.tolist(), ax=ax)
    ax.set_title(f'Top source shifts - {source_name}')
    ax.set_xlabel('Weekly balance range')
    ax.set_ylabel('Source')
    ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.show()


## Conclusion

The next cell prints a concise summary from the computed tables so the notebook remains data-driven rather than narrative-only.


In [ ]:
for source_name, result in analyses.items():
    top_shift = result['source_summary'].iloc[0]
    top_party = result['party_summary'].sort_values('mean_balance', ascending=False).iloc[0]
    print(f'{source_name}: strongest shift source = {top_shift[source_name]}')
    print(f'  balance range: {top_shift.balance_range:.4f}, slope/week: {top_shift.slope_per_week:.4f}, identity ad share: {top_shift.identity_ad_share:.3f}')
    print(f'  party with highest mean balance: {top_party.macro_party_uap} ({top_party.mean_balance:.4f})')
    print()
